# Module 0: Panel Data Preparation and Structure

## Learning Objectives

By completing this notebook, you will:
1. Load and structure panel data with multi-level indexing
2. Convert between long and wide panel data formats
3. Create lagged and differenced variables for panel data
4. Detect balanced vs unbalanced panels
5. Handle missing data in panel contexts

## Prerequisites

- Module 0.1: OLS Fundamentals (completed)
- Pandas DataFrames and indexing
- Basic understanding of time series concepts

## Estimated Time

60-75 minutes

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seed for reproducibility
np.random.seed(42)

print("✅ Setup complete")

## 1. Panel Data Structure: Long vs Wide Format

### What is Panel Data?

Panel data (also called longitudinal data) tracks multiple entities over multiple time periods.

**Notation:**
- $i$ indexes entities (countries, firms, individuals)
- $t$ indexes time periods (years, quarters, months)
- $y_{it}$ is the outcome for entity $i$ at time $t$

### Two Common Formats

**Long Format** (preferred for analysis):
```
entity_id | year | gdp | investment
----------|------|-----|------------
    1     | 2000 | 100 |    20
    1     | 2001 | 105 |    22
    2     | 2000 |  80 |    15
    2     | 2001 |  82 |    16
```

**Wide Format** (easier for humans to read):
```
entity_id | gdp_2000 | gdp_2001 | inv_2000 | inv_2001
----------|----------|----------|----------|----------
    1     |   100    |   105    |    20    |    22
    2     |    80    |    82    |    15    |    16
```

### Create Sample Panel Data

Let's generate data from a simple economic model:

$$\text{GDP}_{it} = \alpha_i + \beta \cdot \text{Investment}_{it} + \gamma \cdot \text{Year}_t + \epsilon_{it}$$

where $\alpha_i$ represents country-specific productivity.

In [ ]:
# Generate panel data
n_entities = 50  # Number of countries
n_periods = 10   # Number of years

# Entity IDs and time periods
entities = np.arange(1, n_entities + 1)
years = np.arange(2010, 2010 + n_periods)

# Create all combinations (balanced panel)
panel_index = pd.MultiIndex.from_product(
    [entities, years],
    names=['country_id', 'year']
)

# Country-specific effects (productivity)
np.random.seed(42)
alpha_i = np.random.randn(n_entities) * 20 + 100

# Generate investment (with some persistence)
investment = []
for i in range(n_entities):
    # AR(1) process for investment
    inv_i = [20 + np.random.randn() * 5]
    for t in range(1, n_periods):
        inv_i.append(0.7 * inv_i[-1] + 0.3 * 20 + np.random.randn() * 3)
    investment.extend(inv_i)

investment = np.array(investment)

# Generate GDP
# Repeat alpha_i for each time period
alpha_repeated = np.repeat(alpha_i, n_periods)

# Year effects (time trend)
year_effect = np.tile(np.arange(n_periods) * 2, n_entities)

# GDP = alpha_i + 0.5*investment + 2*year_trend + error
epsilon = np.random.randn(n_entities * n_periods) * 5
gdp = alpha_repeated + 0.5 * investment + year_effect + epsilon

# Create DataFrame
df_long = pd.DataFrame({
    'gdp': gdp,
    'investment': investment
}, index=panel_index)

# Reset index to make entity_id and year regular columns
df_long_flat = df_long.reset_index()

print("Panel Dataset Created")
print(f"Entities: {n_entities}")
print(f"Time periods: {n_periods}")
print(f"Total observations: {len(df_long)}")
print("\nFirst 10 rows:")
print(df_long_flat.head(10))

## 2. Multi-Level Indexing

Pandas **MultiIndex** is essential for panel data operations. It allows efficient indexing by both entity and time.

In [ ]:
# Set multi-index
df_panel = df_long_flat.set_index(['country_id', 'year'])

print("DataFrame with MultiIndex:")
print(df_panel.head(15))

# Accessing specific entity
print("\n" + "="*50)
print("Data for Country 1:")
print(df_panel.loc[1])

# Accessing specific entity-time
print("\n" + "="*50)
print("GDP for Country 1 in 2015:")
print(df_panel.loc[(1, 2015), 'gdp'])

## 3. Format Conversion: Long to Wide and Back

### Long to Wide

Use `pivot()` or `pivot_table()` to create wide format.

In [ ]:
# Convert to wide format
df_wide = df_long_flat.pivot(
    index='country_id',
    columns='year',
    values=['gdp', 'investment']
)

print("Wide Format (first 5 countries, first 5 years):")
print(df_wide.iloc[:5, :10])

### Wide to Long

Use `melt()` or `stack()` to convert back to long format.

In [ ]:
# Method 1: Using stack()
df_long_restored = df_wide.stack(level='year').reset_index()

print("Restored Long Format:")
print(df_long_restored.head(10))

# Verify it matches original
print(f"\nShape matches: {df_long_restored.shape == df_long_flat.shape}")

## 4. Creating Lagged and Differenced Variables

### Why Lags Matter in Panel Data

Many economic relationships involve dynamics:
- **Lagged dependent variable:** $y_{it} = \rho y_{i,t-1} + X_{it}\beta + \epsilon_{it}$
- **Lagged predictors:** Effect of past investment on current GDP
- **Differences:** Growth rates = $\Delta y_{it} = y_{it} - y_{i,t-1}$

In [ ]:
# Create lagged variables (MUST group by entity first)
df_with_lags = df_long_flat.copy()

# Lag investment by 1 period within each country
df_with_lags['investment_lag1'] = df_with_lags.groupby('country_id')['investment'].shift(1)

# Lag GDP by 1 period
df_with_lags['gdp_lag1'] = df_with_lags.groupby('country_id')['gdp'].shift(1)

# Create first difference (growth rates)
df_with_lags['gdp_growth'] = df_with_lags.groupby('country_id')['gdp'].diff()
df_with_lags['inv_growth'] = df_with_lags.groupby('country_id')['investment'].diff()

# Display
print("Data with Lags and Differences (Country 1):")
print(df_with_lags[df_with_lags['country_id'] == 1].head(8))

# Note: First period has NaN for lags
print("\nNote: First period for each entity has NaN for lagged variables")

### Visualize Lagged Relationship

In [ ]:
# Plot GDP vs lagged investment
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Remove NaN for plotting
plot_data = df_with_lags.dropna(subset=['investment_lag1'])

# Panel 1: GDP vs current investment
axes[0].scatter(plot_data['investment'], plot_data['gdp'], 
                alpha=0.3, s=20)
axes[0].set_xlabel('Investment (t)', fontsize=11)
axes[0].set_ylabel('GDP (t)', fontsize=11)
axes[0].set_title('GDP vs Current Investment', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Panel 2: GDP vs lagged investment
axes[1].scatter(plot_data['investment_lag1'], plot_data['gdp'], 
                alpha=0.3, s=20, color='orange')
axes[1].set_xlabel('Investment (t-1)', fontsize=11)
axes[1].set_ylabel('GDP (t)', fontsize=11)
axes[1].set_title('GDP vs Lagged Investment', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Balanced vs Unbalanced Panels

### Definitions

- **Balanced panel:** Every entity observed in every time period
- **Unbalanced panel:** Some entities missing in some periods

### Detecting Balance

In [ ]:
def check_panel_balance(df, entity_col, time_col):
    """
    Check if panel is balanced.
    
    Parameters
    ----------
    df : pd.DataFrame
        Panel data
    entity_col : str
        Name of entity ID column
    time_col : str
        Name of time column
    
    Returns
    -------
    dict
        Dictionary with balance information
    """
    # Count observations per entity
    counts = df.groupby(entity_col)[time_col].count()
    
    # Check if all equal
    is_balanced = counts.nunique() == 1
    
    # Summary stats
    n_entities = df[entity_col].nunique()
    n_periods = df[time_col].nunique()
    total_obs = len(df)
    expected_obs = n_entities * n_periods
    
    return {
        'is_balanced': is_balanced,
        'n_entities': n_entities,
        'n_periods': n_periods,
        'total_obs': total_obs,
        'expected_obs_if_balanced': expected_obs,
        'obs_per_entity_min': counts.min(),
        'obs_per_entity_max': counts.max(),
        'obs_per_entity_mean': counts.mean()
    }

# Check our panel
balance_info = check_panel_balance(df_long_flat, 'country_id', 'year')

print("Panel Balance Check:")
print("=" * 50)
for key, value in balance_info.items():
    print(f"{key:<30}: {value}")

if balance_info['is_balanced']:
    print("\n✅ Panel is BALANCED")
else:
    print("\n⚠️  Panel is UNBALANCED")

### Create an Unbalanced Panel

Let's randomly drop some observations to simulate missing data.

In [ ]:
# Create unbalanced panel by randomly dropping 10% of observations
np.random.seed(123)
drop_indices = np.random.choice(
    df_long_flat.index, 
    size=int(0.1 * len(df_long_flat)), 
    replace=False
)

df_unbalanced = df_long_flat.drop(drop_indices).reset_index(drop=True)

# Check balance
balance_info_unb = check_panel_balance(df_unbalanced, 'country_id', 'year')

print("Unbalanced Panel:")
print("=" * 50)
for key, value in balance_info_unb.items():
    print(f"{key:<30}: {value}")

# Visualize missing pattern
obs_counts = df_unbalanced.groupby('country_id').size()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(obs_counts.index, obs_counts.values, alpha=0.7)
ax.axhline(y=n_periods, color='red', linestyle='--', 
           label=f'Expected (balanced): {n_periods}')
ax.set_xlabel('Country ID', fontsize=11)
ax.set_ylabel('Number of Observations', fontsize=11)
ax.set_title('Observations per Entity (Unbalanced Panel)', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Within and Between Variation

Panel data contains two types of variation:
- **Within variation:** Changes over time within an entity
- **Between variation:** Differences across entities

Fixed effects uses **within variation only**.

In [ ]:
def decompose_variation(df, entity_col, time_col, var_col):
    """
    Decompose variable into within and between components.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str  
    var_col : str
        Variable to decompose
    
    Returns
    -------
    dict
        Variance decomposition
    """
    # Overall mean
    overall_mean = df[var_col].mean()
    
    # Entity means (average over time for each entity)
    entity_means = df.groupby(entity_col)[var_col].transform('mean')
    
    # Between component: (entity_mean - overall_mean)
    between_component = entity_means - overall_mean
    
    # Within component: (value - entity_mean)
    within_component = df[var_col] - entity_means
    
    # Compute variances
    total_var = df[var_col].var()
    between_var = between_component.var()
    within_var = within_component.var()
    
    return {
        'total_variance': total_var,
        'between_variance': between_var,
        'within_variance': within_var,
        'between_fraction': between_var / total_var,
        'within_fraction': within_var / total_var,
        'within_component': within_component,
        'between_component': between_component
    }

# Decompose GDP variation
gdp_decomp = decompose_variation(df_long_flat, 'country_id', 'year', 'gdp')

print("GDP Variance Decomposition:")
print("=" * 50)
print(f"Total Variance:    {gdp_decomp['total_variance']:.2f}")
print(f"Between Variance:  {gdp_decomp['between_variance']:.2f} ({gdp_decomp['between_fraction']*100:.1f}%)")
print(f"Within Variance:   {gdp_decomp['within_variance']:.2f} ({gdp_decomp['within_fraction']*100:.1f}%)")

# Same for investment
inv_decomp = decompose_variation(df_long_flat, 'country_id', 'year', 'investment')

print("\nInvestment Variance Decomposition:")
print("=" * 50)
print(f"Total Variance:    {inv_decomp['total_variance']:.2f}")
print(f"Between Variance:  {inv_decomp['between_variance']:.2f} ({inv_decomp['between_fraction']*100:.1f}%)")
print(f"Within Variance:   {inv_decomp['within_variance']:.2f} ({inv_decomp['within_fraction']*100:.1f}%)")

### Visualize Decomposition

In [ ]:
# Plot for first 5 countries
sample_countries = [1, 2, 3, 4, 5]
df_sample = df_long_flat[df_long_flat['country_id'].isin(sample_countries)]

fig, ax = plt.subplots(figsize=(12, 6))

# Plot GDP for each country
for country in sample_countries:
    country_data = df_sample[df_sample['country_id'] == country]
    ax.plot(country_data['year'], country_data['gdp'], 
            marker='o', label=f'Country {country}', alpha=0.7)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('GDP', fontsize=12)
ax.set_title('GDP Over Time: Within and Between Variation', fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

# Add text annotation
ax.text(0.98, 0.05, 
        'Within variation: Changes over time\nBetween variation: Differences in levels',
        transform=ax.transAxes, fontsize=10,
        verticalalignment='bottom', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.show()

---

## Exercises

### Exercise 1: Create Multi-Period Lags

**Task:** Write a function that creates multiple lags (1, 2, 3 periods) of a variable.

**Expected Output:** DataFrame with columns `var_lag1`, `var_lag2`, `var_lag3`

**Hints:**
- Use `groupby()` with `shift()`
- Loop over lag periods or use a comprehension

In [ ]:
# YOUR CODE HERE
def create_lags(df, entity_col, var_col, max_lag=3):
    """
    Create multiple lags of a variable.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
        Entity identifier column
    var_col : str
        Variable to lag
    max_lag : int
        Maximum number of lags to create
    
    Returns
    -------
    pd.DataFrame
        DataFrame with original column and lag columns
    """
    # TODO: Implement this function
    pass

In [ ]:
# SOLUTION (hidden)
def create_lags_solution(df, entity_col, var_col, max_lag=3):
    """Create multiple lags of a variable."""
    result = df.copy()
    
    for lag in range(1, max_lag + 1):
        lag_col = f"{var_col}_lag{lag}"
        result[lag_col] = result.groupby(entity_col)[var_col].shift(lag)
    
    return result

In [ ]:
# Test your function
def test_exercise_1():
    """Test lag creation."""
    result = create_lags(df_long_flat, 'country_id', 'gdp', max_lag=3)
    
    assert result is not None, "Function returns None - did you implement it?"
    
    # Check columns exist
    expected_cols = ['gdp_lag1', 'gdp_lag2', 'gdp_lag3']
    for col in expected_cols:
        assert col in result.columns, f"Missing column: {col}"
    
    # Check lag values are correct
    country_1 = result[result['country_id'] == 1].reset_index(drop=True)
    
    # gdp_lag1 at t=1 should equal gdp at t=0
    assert pd.isna(country_1.loc[0, 'gdp_lag1']), "First period lag1 should be NaN"
    assert np.isclose(country_1.loc[1, 'gdp_lag1'], country_1.loc[0, 'gdp']), \
        "Lag1 value incorrect"
    
    # gdp_lag2 at t=2 should equal gdp at t=0
    assert np.isclose(country_1.loc[2, 'gdp_lag2'], country_1.loc[0, 'gdp']), \
        "Lag2 value incorrect"
    
    print("✅ Correct! Lag columns created successfully.")
    print("\nExample (Country 1):")
    print(country_1[['year', 'gdp', 'gdp_lag1', 'gdp_lag2', 'gdp_lag3']].head(6))
    return True

# Uncomment to test
# test_exercise_1()

### Exercise 2: Identify Entities with Missing Periods

**Task:** Write a function that returns which entities have missing time periods in an unbalanced panel.

**Expected Output:** List of entity IDs that don't have all time periods

**Hints:**
- Group by entity and count observations
- Compare to the maximum number of periods observed

In [ ]:
# YOUR CODE HERE
def find_incomplete_entities(df, entity_col, time_col):
    """
    Find entities with missing time periods.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    
    Returns
    -------
    list
        Entity IDs with missing periods
    """
    # TODO: Implement this function
    pass

In [ ]:
# SOLUTION (hidden)
def find_incomplete_entities_solution(df, entity_col, time_col):
    """Find entities with missing time periods."""
    # Count observations per entity
    obs_counts = df.groupby(entity_col)[time_col].count()
    
    # Expected count (maximum observed)
    max_periods = df[time_col].nunique()
    
    # Find entities with fewer than max
    incomplete = obs_counts[obs_counts < max_periods].index.tolist()
    
    return incomplete

In [ ]:
# Test your function
def test_exercise_2():
    """Test incomplete entity detection."""
    # Test on unbalanced panel
    incomplete = find_incomplete_entities(df_unbalanced, 'country_id', 'year')
    
    assert incomplete is not None, "Function returns None - did you implement it?"
    assert isinstance(incomplete, list), "Should return a list"
    assert len(incomplete) > 0, "Unbalanced panel should have some incomplete entities"
    
    # Test on balanced panel (should be empty)
    incomplete_balanced = find_incomplete_entities(df_long_flat, 'country_id', 'year')
    assert len(incomplete_balanced) == 0, "Balanced panel should have no incomplete entities"
    
    print(f"✅ Correct! Found {len(incomplete)} incomplete entities in unbalanced panel.")
    print(f"Examples: {incomplete[:5]}")
    print(f"\nBalanced panel has {len(incomplete_balanced)} incomplete entities (as expected).")
    return True

# Uncomment to test
# test_exercise_2()

### Exercise 3: Compute Within-Entity Summary Statistics

**Task:** For each entity, compute the mean, standard deviation, min, and max of a variable over time.

**Expected Output:** DataFrame with entity_id and summary statistics columns

**Hints:**
- Use `groupby()` with `.agg()`
- Can pass multiple functions as a list

In [ ]:
# YOUR CODE HERE
def entity_summary_stats(df, entity_col, var_col):
    """
    Compute summary statistics within each entity.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    var_col : str
        Variable to summarize
    
    Returns
    -------
    pd.DataFrame
        Summary statistics by entity
    """
    # TODO: Implement this function
    pass

In [ ]:
# SOLUTION (hidden)
def entity_summary_stats_solution(df, entity_col, var_col):
    """Compute summary statistics within each entity."""
    summary = df.groupby(entity_col)[var_col].agg([
        ('mean', 'mean'),
        ('std', 'std'),
        ('min', 'min'),
        ('max', 'max')
    ]).reset_index()
    
    return summary

In [ ]:
# Test your function
def test_exercise_3():
    """Test entity summary statistics."""
    summary = entity_summary_stats(df_long_flat, 'country_id', 'gdp')
    
    assert summary is not None, "Function returns None - did you implement it?"
    assert len(summary) == n_entities, f"Should have {n_entities} rows"
    
    # Check columns
    expected_cols = ['mean', 'std', 'min', 'max']
    for col in expected_cols:
        assert col in summary.columns, f"Missing column: {col}"
    
    # Check values are reasonable
    assert (summary['min'] <= summary['mean']).all(), "Min should be <= mean"
    assert (summary['mean'] <= summary['max']).all(), "Mean should be <= max"
    assert (summary['std'] >= 0).all(), "Std should be non-negative"
    
    print("✅ Correct! Summary statistics computed successfully.")
    print("\nFirst 10 entities:")
    print(summary.head(10))
    return True

# Uncomment to test
# test_exercise_3()

---

## Summary

### Key Takeaways

1. **Panel Structure:** Use MultiIndex with (entity, time) for efficient operations

2. **Long vs Wide:** Long format is preferred for analysis; use `pivot()` and `stack()`/`melt()` to convert

3. **Lagged Variables:** Always use `groupby(entity).shift()` to respect panel structure

4. **Balanced Panels:** Check balance with group counts; unbalanced panels are common in practice

5. **Within vs Between:** Fixed effects exploits within variation; random effects uses both

### Connections to Panel Methods

- **Fixed Effects:** Applies within transformation (demeaning by entity)
- **Random Effects:** Uses weighted average of within and between variation
- **First Differences:** Uses `diff()` instead of demeaning
- **Dynamic Models:** Use lagged dependent variables as predictors

### What's Next

In Module 1, we'll start with pooled OLS on panel data and understand why it can be biased when entity effects are present.

---

**Next:** Module 1 - Panel Structure and Pooled OLS